In [1]:
import json
import matplotlib.pyplot as plt
import numpy as np
import os

In [2]:
# Plot
plot_directory = 'result/plot_set'

In [3]:
# colors = [
#     '#e6194B',
#     '#f58231',
#     '#9A6324',
#     '#911eb4',
#     '#3cb44b',
#     '#f032e6',
#     '#4363d8',
# ]

In [4]:
colors = [
	'#FF0000',
	'#00FFFF',
	'#0000FF',
	'#00008B',
	'#ADD8E6',
	'#800080',
	'#7FFFD4',
	'#008000',
	'#FF00FF',
	'#FFC0CB',
	'#C0C0C0',
	'#FFA500',
	'#000000',
	'#800000',
]

In [5]:
def load_json_file(file_path):
	try:
		with open(file_path, 'r') as file:
			data = json.load(file)
		return data
	
	except Exception as e:
		print(f"An error occurred while loading the JSON file: {e}")
		return None

In [6]:
def extract_fpss(metric_list):
	return list(metric_list[list(metric_list.keys())[0]][0]['metric'].keys())

In [7]:
def to_accuracy_vector(accuracy_result_seq, fpss, type='F1'):
	accuracy_vector = []
	for fps in fpss:
		accuracy_vector.append(accuracy_result_seq[fps][type])
	
	return accuracy_vector

In [8]:
def plot_scatter(xs, ys, x_label, y_label, title, fig_size=(8, 8)):
	plt.figure(figsize=fig_size)
	plt.scatter(xs, ys, c=colors[0], s=30, alpha=0.5)
	plt.title(title)
	plt.xlabel(x_label)
	plt.ylabel(y_label)
	plt.show()

In [9]:
def plot_scatter_label(xs, ys, labels, x_label, y_label, title, fig_size=(8, 8)):
	plt.figure(figsize=fig_size)
	for i in range(len(xs)):
		plt.scatter(xs[i], ys[i], c=colors[labels[i]], s=30, alpha=0.5)
	plt.title(title)
	plt.xlabel(x_label)
	plt.ylabel(y_label)
	plt.show()

In [10]:
def filter_list_index(input_list, indices_to_remove):
	return [item for i, item in enumerate(input_list) if i not in indices_to_remove]

In [11]:
def round_float_to_sigfigs(number, sigfigs):
	return round(number, sigfigs)

## Plot

In [12]:
omv_features = ["Left-Top", "Right-Top", "Left-Bottom", "Right-Bottom", "Object-Amount", "Confidence", "IOU"]

In [13]:
plot_filenames = sorted(os.listdir(plot_directory))
plot_video_names = sorted(list(set([f.split('_')[0] for f in plot_filenames])))

In [14]:
fpss = extract_fpss(load_json_file(os.path.join(plot_directory, plot_video_names[0] + "_Accuracy_Result.json")))
# fpss = ['2', '3', '5', '6', '10', '15']

In [15]:
omv_videos = []
acc_videos = []

for v in plot_video_names:
	omv_dict = {}
	for fps in fpss:
		omv_dict[fps] = []
	acc_list = []

	accuracy_result = load_json_file(os.path.join(plot_directory, v + "_Accuracy_Result.json"))
	movement_result = load_json_file(os.path.join(plot_directory, v + "_Movement_Result.json"))

	for class_idx in list(accuracy_result.keys()):
		for i in range(len(accuracy_result[class_idx])):
			accuracy_vector = to_accuracy_vector(accuracy_result[class_idx][i]['metric'], fpss)
			acc_list.append(accuracy_vector)

			for fps in fpss:
				movement_vector = movement_result[class_idx][i]['movement'][fps]
				omv_dict[fps].append(movement_vector)
	
	omv_videos.append(omv_dict)
	acc_videos.append(acc_list)

In [16]:
# Corr Plot Format 1

for i in range(len(plot_video_names)):
	video_name = plot_video_names[i]
	print(omv_videos[0][fpss[0]])
	for j in range(len(omv_videos[0][fpss[0]][0])):
		for k in range(len(fpss)):
			fps = fpss[k]
			
			omv_fps = list(np.array(omv_videos[i][fps])[:, j])
			acc_fps = list(np.array(acc_videos[i])[:, k])

			# Remove Outliers
			outlier_index = [l for l in range(len(omv_fps)) if omv_fps[l] == -1]
			omv_fps_clean = filter_list_index(omv_fps, outlier_index)
			acc_fps_clean = filter_list_index(acc_fps, outlier_index)

			title = f'{video_name}; OMV Feature {j} ({omv_features[j]}); FPS: {fps}'

			correlation_matrix = np.corrcoef(omv_fps_clean, acc_fps_clean)
			correlation_coefficient = correlation_matrix[0, 1]
			print(f"{title} -> Corr: {round_float_to_sigfigs(correlation_coefficient, 3)}")
			# plot_scatter(omv_fps_clean, acc_fps_clean, 'OMV Feature', 'ACC', title)

		print("")

[[0.4280800882118778, 0.39429983042007044, 0.42837352043403026, 0.39461910068916695, 4.0, 0.439090125, 0.6811701603160017], [0.4280800882118778, 0.39429983042007044, 0.42837352043403026, 0.39461910068916695, 4.0, 0.439090125, 0.6811701603160017], [0.34198288158577644, 0.3366421144967209, 0.3420411844081877, 0.33671219524139845, 3.0, 0.40760766666666665, 0.6779533401586577], [0.18025532047480883, 0.17118939551255605, 0.18085424620954324, 0.17199785653924446, 3.0, 0.41280866666666666, 0.5943175780605695], [0.15903023662899252, 0.1610191301151917, 0.16072680009658163, 0.16176690509109534, 2.0, 0.45141575, 0.6952166054197992], [0.8943952754609844, 0.8610105748357159, 0.8942828831060842, 0.86089382401579, 1.0, 0.4844445, 0.90907183721509], [0.1412231158091692, 0.12176282740003207, 0.1410624135482234, 0.12157640483930257, 1.0, 0.369685, 0.9086792587988065], [0.7670518714552153, 0.7375492082261006, 0.7670133262383603, 0.7375091825725046, 2.0, 0.33023975, 0.8802943575924249], [0.13103123426278

In [17]:
# Corr Plot Format 2
np_result = np.zeros((len(fpss), len(omv_videos[0][fpss[0]][0])))

for k in range(len(fpss)):
	fps = fpss[k]
	print(f"FPS: {fps}")

	for j in range(len(omv_videos[0][fpss[0]][0])):
		print(f"OMV Feature {j} ({omv_features[j]})")
		corr_list = []

		for i in range(len(plot_video_names)):
			video_name = plot_video_names[i]
		
			omv_fps = list(np.array(omv_videos[i][fps])[:, j])
			acc_fps = list(np.array(acc_videos[i])[:, k])

			# Remove Outliers
			outlier_index = [l for l in range(len(omv_fps)) if omv_fps[l] == -1]
			omv_fps_clean = filter_list_index(omv_fps, outlier_index)
			acc_fps_clean = filter_list_index(acc_fps, outlier_index)

			correlation_matrix = np.corrcoef(omv_fps_clean, acc_fps_clean)
			correlation_coefficient = correlation_matrix[0, 1]
			corr_list.append(correlation_coefficient)

		print(f'Average Corr Among All {len(plot_video_names)} Videos {round_float_to_sigfigs(np.average(np.array(corr_list)), 3)}')
		np_result[k][j] = round_float_to_sigfigs(np.average(np.array(corr_list)), 3)

		print("")

FPS: 1
OMV Feature 0 (Left-Top)
Average Corr Among All 4 Videos -0.062

OMV Feature 1 (Right-Top)
Average Corr Among All 4 Videos -0.067

OMV Feature 2 (Left-Bottom)
Average Corr Among All 4 Videos -0.064

OMV Feature 3 (Right-Bottom)
Average Corr Among All 4 Videos -0.067

OMV Feature 4 (Object-Amount)
Average Corr Among All 4 Videos 0.376

OMV Feature 5 (Confidence)
Average Corr Among All 4 Videos 0.151

OMV Feature 6 (IOU)
Average Corr Among All 4 Videos 0.453

FPS: 2
OMV Feature 0 (Left-Top)
Average Corr Among All 4 Videos -0.023

OMV Feature 1 (Right-Top)
Average Corr Among All 4 Videos -0.026

OMV Feature 2 (Left-Bottom)
Average Corr Among All 4 Videos -0.031

OMV Feature 3 (Right-Bottom)
Average Corr Among All 4 Videos -0.03

OMV Feature 4 (Object-Amount)
Average Corr Among All 4 Videos 0.383

OMV Feature 5 (Confidence)
Average Corr Among All 4 Videos 0.186

OMV Feature 6 (IOU)
Average Corr Among All 4 Videos 0.398

FPS: 3
OMV Feature 0 (Left-Top)
Average Corr Among All 4 Videos

In [18]:
print(np_result)

[[-0.062 -0.067 -0.064 -0.067  0.376  0.151  0.453]
 [-0.023 -0.026 -0.031 -0.03   0.383  0.186  0.398]
 [-0.056 -0.05  -0.061 -0.052  0.35   0.245  0.29 ]
 [ 0.041  0.054  0.04   0.054  0.17   0.334  0.328]
 [ 0.076  0.101  0.073  0.097  0.105  0.385  0.34 ]
 [ 0.082  0.104  0.083  0.105 -0.018  0.437  0.24 ]
 [ 0.059  0.08   0.06   0.08  -0.074  0.459  0.18 ]
 [ 0.067  0.079  0.07   0.082 -0.109  0.483  0.133]]
